# GPT od podstaw

Warsztaty w ramach Funduszu Zdolni (30 kwietnia - 2 maja 2026).

## Notebook 4: Samouwaga (self-attention)

**Inspiracja:** [Let's build GPT - Andrej Karpathy (YouTube)](https://www.youtube.com/watch?v=kCc8FmEb1nY) oraz [nanoGPT](https://github.com/karpathy/nanoGPT). Oryginalny artykuł: [Attention Is All You Need (2017)](https://arxiv.org/abs/1706.03762).

### Po co nam to?

MLP z [Notebooka 3](3_pytorch_network.ipynb) patrzył na 3 znaki wstecz. Skleił je w jeden długi wektor i przepuścił przez warstwę ukrytą. Problem: jeśli chcemy patrzeć na 32 albo 256 znaków wstecz, ten sklejony wektor robi się ogromny, a model musi się od zera nauczyć, że pozycje bliżej końca są ważniejsze.

**Pomysł uwagi**: niech każdy znak sam zdecyduje, na które poprzednie znaki popatrzeć i ile uwagi im poświęcić. To serce transformera (i całego nowoczesnego AI).

### Krok 1: średnia ważona poprzednich tokenów

Najprostsza wersja: każdy token na pozycji `t` patrzy na siebie i wszystkie poprzednie i bierze ich **średnią**.

Można to zapisać jako mnożenie macierzowe `W @ X`, gdzie `W` to macierz wag, np. dla 4 tokenów:

```
W = [[1.00, 0.00, 0.00, 0.00],
     [0.50, 0.50, 0.00, 0.00],
     [0.33, 0.33, 0.33, 0.00],
     [0.25, 0.25, 0.25, 0.25]]
```

Każdy wiersz to "przepis", jak token na danej pozycji łączy wszystkie poprzednie. Dwie ważne rzeczy:
- **wiersze sumują się do 1** (to są prawdopodobieństwa),
- **macierz jest dolnotrójkątna** — token nie widzi przyszłości (to *causal mask*).

Średnia z równymi wagami nie jest jednak mądra — traktuje wszystkie poprzednie tokeny tak samo. A gdyby tak każdy token mógł sam wybierać, na co chce patrzeć?

### Krok 2: query, key, value — skąd biorą się wagi

Każdy token tworzy z embeddingu trzy wektory:

- **query (Q)** — *"czego szukam?"* (np. "szukam czasownika")
- **key (K)** — *"co oferuję?"* (np. "jestem rzeczownikiem")
- **value (V)** — *"co przekażę dalej, jeśli mnie wybierzesz?"*

Analogia z wyszukiwarką: piszesz **query** w Google, dokumenty mają tytuły (**keys**), a same dokumenty to **values**. Google liczy podobieństwo `query · key` i pokazuje top-k wartości.

W uwadze robimy podobnie, tylko nie wybieramy top-k a robimy ważoną sumę:
1. Liczymy *iloczyn skalarny* `Q · K` dla każdej pary tokenów. Wysoki iloczyn = pasują do siebie.
2. Stosujemy maskę trójkątną (token nie widzi przyszłości).
3. **Softmax** zamienia liczby w prawdopodobieństwa (sumują się do 1).
4. Mnożymy te wagi przez wartości V.

**Najważniejsze**: wagi są **danymi-zależne**. Inny tekst → inne Q i K → inne wagi. Sieć sama uczy się, na co patrzeć.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from livelossplot import PlotLosses

torch.manual_seed(42)

### Dane

Tak samo jak w notebooku 3 — czytamy *Pana Tadeusza*, robimy słownik znaków, dzielimy 90/10. Tylko zwiększamy `block_size` (długość kontekstu, transformerowa nazwa).

In [ ]:
with open("data/pan-tadeusz.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for i, c in enumerate(chars)}

data = torch.tensor([char2id[c] for c in text], dtype=torch.long)
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
test_data = data[split_idx:]

block_size = 16  # ile znaków wstecz model widzi (kontekst)
batch_size = 64

def losuj_paczke(split: str = "train") -> tuple[torch.Tensor, torch.Tensor]:
    zrodlo = train_data if split == "train" else test_data
    ix = torch.randint(0, len(zrodlo) - block_size - 1, (batch_size,))
    x = torch.stack([zrodlo[i:i+block_size] for i in ix])
    y = torch.stack([zrodlo[i+1:i+block_size+1] for i in ix])
    return x, y

print(f"Słownik: {vocab_size} znaków")

### Klasa `Head` — jedna głowa uwagi w kodzie

Wszystkie cztery elementy z naszej intuicji w 8 linijkach kodu:
1. trzy warstwy liniowe — Q, K, V (każda projekuje embedding na inną przestrzeń),
2. iloczyn skalarny `Q · K` daje surowe wagi,
3. maska trójkątna zeruje (przez `-inf`) wagi do przyszłości,
4. softmax + mnożenie przez V.

Dzielenie przez `√head_size` to drobny trick numeryczny — bez tego softmax saturuje się przy większych wymiarach.

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd: int, head_size: int, block_size: int):
        super().__init__()
        # Trzy projekcje liniowe — każda "patrzy" na embedding inaczej
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Maska trójkątna - nie parametr, ale bufor (jedzie z modelem na GPU)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.query(x)              # "czego szukam"  (B, T, head_size)
        k = self.key(x)                # "co oferuję"
        v = self.value(x)              # "co przekażę"
        # Wagi: jak bardzo każda para (i, j) do siebie pasuje
        wagi = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)         # (B, T, T)
        # Nie patrz w przyszłość
        wagi = wagi.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        # Softmax robi z tego rozkład prawdopodobieństwa po każdym wierszu
        wagi = F.softmax(wagi, dim=-1)
        # Ważona suma wartości
        return wagi @ v                                                  # (B, T, head_size)

### Model: embedding tokena + embedding pozycji + jedna głowa uwagi

Sama uwaga jest "bez kolejności" — gdybyśmy podali `[a, b, c]` albo `[c, b, a]`, dostalibyśmy te same wagi (jeszcze bez maski). Dlatego do embeddingu tokena dodajemy **embedding pozycji** — osobny wektor dla każdej pozycji 0, 1, …, block_size−1. Po dodaniu sieć już wie, *gdzie* dany znak stoi.

In [ ]:
class JednoglowyModel(nn.Module):
    def __init__(self, vocab_size: int, n_embd: int, head_size: int, block_size: int):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.head = Head(n_embd, head_size, block_size)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T))   # (B, T, n_embd)
        x = self.head(x)                                            # (B, T, head_size)
        return self.lm_head(x)                                      # (B, T, vocab_size)

    @torch.no_grad()
    def generuj(self, idx: torch.Tensor, max_nowych: int, temperatura: float = 1.0) -> torch.Tensor:
        for _ in range(max_nowych):
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)[:, -1, :] / temperatura
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

In [ ]:
model = JednoglowyModel(vocab_size, n_embd=32, head_size=32, block_size=block_size)
print(f"Liczba parametrów: {sum(p.numel() for p in model.parameters()):,}")

### Trening

Wyjście modelu ma kształt `(B, T, vocab_size)` — przewidujemy następny znak na **każdej** pozycji jednocześnie. Dlatego spłaszczamy do `(B*T, vocab_size)` przed cross-entropy. Jedna paczka długości 16 daje nam więc 16 "przykładów" naraz.

In [ ]:
def oblicz_strate(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    logits = model(x)
    B, T, V = logits.shape
    return F.cross_entropy(logits.view(B*T, V), y.view(B*T))

@torch.no_grad()
def strata_test() -> float:
    model.eval()
    s = sum(oblicz_strate(*losuj_paczke("test")).item() for _ in range(20)) / 20
    model.train()
    return s

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
liveloss = PlotLosses()

for it in range(2000):
    xb, yb = losuj_paczke("train")
    loss = oblicz_strate(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if it % 50 == 0:
        liveloss.update({"loss": loss.item(), "val_loss": strata_test()})
        liveloss.send()

### Generowanie tekstu

In [ ]:
def generuj_tekst(start_str: str, dlugosc: int = 300, temperatura: float = 1.0) -> str:
    idx = torch.tensor([[char2id[c] for c in start_str]], dtype=torch.long)
    out = model.generuj(idx, max_nowych=dlugosc, temperatura=temperatura)
    return "".join(id2char[i.item()] for i in out[0])

print(generuj_tekst("Litwo, ojczyzno moja", dlugosc=400, temperatura=0.8))

### Bonus: zobaczmy, na co model patrzy

Wagi `wagi` z naszej głowy uwagi to dolnotrójkątna macierz `(T, T)`, w której wiersz `t` mówi, ile uwagi token na pozycji `t` poświęca każdemu z poprzednich. Wyświetlmy ją.

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    # Bierzemy mały kawałek tekstu jako przykład
    przyklad = "Litwo, ojczyzno"[:block_size]
    idx = torch.tensor([[char2id[c] for c in przyklad]])
    x = model.token_emb(idx) + model.pos_emb(torch.arange(idx.shape[1]))
    
    # Liczymy wagi tak jak w Head, ale chcemy je podejrzeć
    h = model.head
    q, k = h.query(x), h.key(x)
    wagi = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
    wagi = wagi.masked_fill(h.tril[:idx.shape[1], :idx.shape[1]] == 0, float("-inf"))
    wagi = F.softmax(wagi, dim=-1)[0]  # (T, T)

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(wagi.numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(przyklad)), list(przyklad))
ax.set_yticks(range(len(przyklad)), list(przyklad))
ax.set_xlabel("na co patrzy →")
ax.set_ylabel("który token (←)")
ax.set_title("Macierz wag uwagi (im ciemniej, tym więcej uwagi)")
plt.tight_layout()

Co widać na obrazku: macierz jest dolnotrójkątna (maska działa), a w poszczególnych wierszach widać wzorce, których model się nauczył (najczęściej najwięcej uwagi do bezpośrednio poprzedzających znaków).

---

**Co dalej?** Jedna głowa to za mało. W [Notebooku 5](5_mini_gpt.ipynb) dorobimy: wiele głów uwagi (każda patrzy na coś innego), warstwy feed-forward, połączenia rezydualne, LayerNorm i kilka bloków jeden na drugim. To już pełny mini-GPT.